# 02 — Fine-Tuning Mistral 7B with QLoRA

**What:** Fine-tune Mistral 7B on invoice field extraction using QLoRA (4-bit quantization + LoRA adapters).

**Where:** Kaggle T4 GPU (16GB VRAM) or Google Colab T4.

**How long:** ~3-4 hours for 3 epochs on ~1000 samples.

**Key decisions (from data analysis):**
- `max_seq_length=768` — 99%+ of our samples are under 700 tokens
- `fp16=True` with no-op GradScaler — T4 is optimized for fp16, bitsandbytes produces bf16 gradients that crash the real GradScaler
- `batch_size=2, grad_accum=8` — effective batch of 16, fits T4 VRAM
- `LoRA r=64, alpha=128` — targets all attention + MLP layers

## 1. Install Dependencies

In [ ]:
!pip install -q peft trl bitsandbytes wandb sentencepiece pydantic rapidfuzz

## 2. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 3. Clone Code & Upload Data

In [ ]:
!git clone https://github.com/Pushparaj13811/invoice-extraction-mistral-7b-fine-tuning.git

import sys
sys.path.insert(0, '/kaggle/working/invoice-extraction-mistral-7b-fine-tuning')

Set API tokens:

In [ ]:
import os

# Set your tokens here
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
os.environ['WANDB_API_KEY'] = 'YOUR_WANDB_KEY'

In [ ]:
import wandb
wandb.login()
print('W&B logged in')

## 4. Configure Training

These hyperparameters are chosen based on our data analysis:
- **LoRA rank 64**: Targets all attention + MLP projection layers (167M trainable / 7.4B total = 2.26%)
- **max_seq_length 768**: Our data analysis showed 99th percentile at ~689 tokens
- **Effective batch size 16**: batch_size=2 * grad_accum=8 — balances VRAM and training stability
- **Learning rate 2e-4**: Standard for QLoRA fine-tuning with cosine schedule

In [ ]:
from src.training.config import TrainingConfig

config = TrainingConfig(
    model_name='mistralai/Mistral-7B-v0.3',
    lora_r=64,
    lora_alpha=128,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_seq_length=768,
)

In [ ]:
print(f'Model:          {config.model_name}')
print(f'LoRA:           r={config.lora_r}, alpha={config.lora_alpha}')
print(f'Batch size:     {config.per_device_train_batch_size} x {config.gradient_accumulation_steps} = {config.effective_batch_size}')
print(f'Epochs:         {config.num_train_epochs}')
print(f'Max seq length: {config.max_seq_length}')
print(f'Learning rate:  {config.learning_rate}')

## 5. Verify Data

Make sure the training data files are accessible:

In [ ]:
# Update these paths based on your setup:
# - Kaggle: '/kaggle/input/invoice-extraction-dataset/train.jsonl'
# - Colab: '/content/data/train.jsonl'

TRAIN_PATH = '/kaggle/input/datasets/pushparajmehta/invoice-extraction-dataset/train.jsonl'
EVAL_PATH = '/kaggle/input/datasets/pushparajmehta/invoice-extraction-dataset/eval.jsonl'

In [ ]:
from src.data.format import load_jsonl

train_data = load_jsonl(TRAIN_PATH)
eval_data = load_jsonl(EVAL_PATH)
print(f'Train samples: {len(train_data)}')
print(f'Eval samples:  {len(eval_data)}')

## 6. Train

This loads the model in 4-bit (QLoRA), applies LoRA adapters, and trains with SFTTrainer.
Training logs are synced to W&B for monitoring.

In [ ]:
from src.training.train import train

trainer = train(
    config=config,
    train_path=TRAIN_PATH,
    eval_path=EVAL_PATH,
    output_dir='/kaggle/working/outputs/',
)
print('Training complete!')

## 7. Verify Saved Adapter

Check that the LoRA adapter weights were saved correctly:

In [ ]:
adapter_path = '/kaggle/working/outputs/final_adapter'
files = os.listdir(adapter_path)
print(f'Adapter files: {files}')

## 8. Quick Sanity Check

Run one test invoice through the fine-tuned model to verify it produces valid JSON:

In [ ]:
from src.evaluation.evaluate import load_finetuned_model
import torch

model, tokenizer = load_finetuned_model(config.model_name, adapter_path)

In [ ]:
test_input = """Invoice #TEST-001
From: Test Corp, 123 Main St
Date: 2024-06-15
Due: 2024-07-15
Item: Widget x2 @ $50.00 = $100.00
Tax: $10.00
Total: $110.00
Currency: USD"""

In [ ]:
prompt = f'### Instruction:\nExtract all invoice fields from the following invoice text as JSON.\n\n### Input:\n{test_input}\n\n### Response:\n'

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

## 9. Save for Download

Copy the adapter to the output directory so you can download it from Kaggle's Output tab:

In [ ]:
import shutil

shutil.copytree(adapter_path, '/kaggle/working/final_adapter')
print('Adapter ready for download from Output tab.')

**Next:** Run `03_evaluation.ipynb` to benchmark against GPT-4o-mini baseline.